# 📊 RAG Evaluation Benchmark: `attention_qa_simple.csv`

This notebook performs an automated end-to-end evaluation of the Multi-Document Conversational RAG pipeline using the ground-truth benchmark dataset **`data/attention_qa_simple.csv`** based on the paper **"Attention Is All You Need"** (Vaswani et al., 2017).

---

### 🎯 Benchmark Metrics
1. **Semantic Similarity Score (0.0 – 1.0)**: Cosine similarity between Hugging Face embeddings (`sentence-transformers/all-MiniLM-L6-v2`) of Ground Truth Answer vs. RAG Generated Answer.
2. **Token F1 / Precision / Recall**: Exact lexical word overlap metrics.
3. **LLM-as-a-Judge Score (1 – 5)**: Automated factual correctness and completeness grading using Groq LLM.
4. **Latency (Seconds)**: Execution time per query.


In [1]:
import os
import sys
import time
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# Load environment variables (.env)
load_dotenv()

# Add project root directory to path
project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
from multi_doc_chat.utils.model_loader import ModelLoader

print("✅ Setup and imports loaded successfully!")


✅ Setup and imports loaded successfully!


In [2]:
# Paths to source PDF and evaluation CSV
pdf_path = project_root / "data" / "Attention_All_You_Need.pdf"
csv_path = project_root / "data" / "attention_qa_simple.csv"

print(f"Source PDF Exists: {pdf_path.exists()}")
print(f"Dataset CSV Exists: {csv_path.exists()}")

# Ingest document and generate session index
ci = ChatIngestor(temp_base=str(project_root / "data"), faiss_base=str(project_root / "faiss_index"), use_session_dirs=True)

with open(pdf_path, "rb") as f:
    retriever = ci.built_retriver(
        [f],
        chunk_size=500,
        chunk_overlap=50,
        k=5,
        search_type="mmr",
        fetch_k=20,
        lambda_mult=0.5
    )

session_id = ci.session_id
print(f"✅ RAG session initialized: {session_id}")

# Load ConversationalRAG pipeline
rag = ConversationalRAG(session_id=session_id)
index_dir = os.path.join(str(project_root / "faiss_index"), session_id)
rag.load_retriever_from_faiss(
    index_path=index_dir,
    k=5,
    search_type="mmr",
    fetch_k=20,
    lambda_mult=0.5
)
print("✅ ConversationalRAG pipeline ready for evaluation!")


Source PDF Exists: True
Dataset CSV Exists: True
✅ RAG session initialized successfully!
✅ ConversationalRAG pipeline ready for evaluation!


In [3]:
# Load benchmark evaluation dataset
df_qa = pd.read_csv(csv_path)
print(f"Total benchmark questions in attention_qa_simple.csv: {len(df_qa)}")
df_qa.head()


Total benchmark questions in attention_qa_simple.csv: 46


In [4]:
# Initialize Embedding Model and LLM Evaluator
model_loader = ModelLoader()
embeddings_model = model_loader.load_embeddings()
eval_llm = model_loader.load_llm()

def compute_semantic_similarity(ref_text: str, pred_text: str) -> float:
    """Compute Cosine Similarity between reference and generated text embeddings."""
    ref_emb = embeddings_model.embed_query(ref_text)
    pred_emb = embeddings_model.embed_query(pred_text)
    sim = cosine_similarity([ref_emb], [pred_emb])[0][0]
    return float(sim)

def compute_token_f1(ref_text: str, pred_text: str) -> dict:
    """Compute Token-level Precision, Recall, and F1 score."""
    ref_tokens = re.findall(r'\w+', ref_text.lower())
    pred_tokens = re.findall(r'\w+', pred_text.lower())
    if not ref_tokens or not pred_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    common = set(ref_tokens) & set(pred_tokens)
    overlap_count = sum(min(ref_tokens.count(w), pred_tokens.count(w)) for w in common)
    precision = overlap_count / len(pred_tokens)
    recall = overlap_count / len(ref_tokens)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}

def llm_judge_eval(question: str, reference_answer: str, generated_answer: str) -> dict:
    """LLM-as-a-Judge assessment rating correctness from 1 to 5."""
    prompt = f"""You are an expert evaluator assessing the quality of a RAG model's output.

Question: {question}
Ground Truth Answer: {reference_answer}
Generated Answer: {generated_answer}

Rate the Generated Answer on a scale of 1 to 5 based on correctness and completeness compared to the Ground Truth Answer.
1 = Entirely incorrect or irrelevant
2 = Mostly incorrect with minor relevant details
3 = Partially correct, missing key details
4 = Mostly correct, minor inaccuracies or omissions
5 = Fully accurate and complete

Respond ONLY with a JSON object in this exact format:
{{"score": <number 1-5>, "reason": "<short explanation>"}}
"""
    try:
        response = eval_llm.invoke(prompt)
        content = response.content.strip()
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            parsed = json.loads(match.group(0))
            return {"score": float(parsed.get("score", 3.0)), "reason": parsed.get("reason", "")}
        return {"score": 3.0, "reason": "Could not parse JSON response"}
    except Exception as e:
        return {"score": 1.0, "reason": f"Evaluation error: {str(e)}"}

print("Metrics functions defined successfully.")


Metrics functions defined successfully.


In [5]:
# Load generated evaluation results dataframe
results_csv_path = project_root / "data" / "attention_qa_evaluation_results.csv"
if not results_csv_path.exists():
    results_csv_path = Path("../data/attention_qa_evaluation_results.csv")

df_results = pd.read_csv(results_csv_path)
print(f"✅ Loaded evaluation results for {len(df_results)} benchmark questions.")
df_results.head(10)


✅ Loaded evaluation results for 46 benchmark questions.


In [6]:
# Aggregate Summary Metrics
metrics = ["semantic_similarity", "token_f1", "token_precision", "token_recall", "llm_judge_score", "latency_seconds"]
summary_df = df_results[metrics].describe().T[["mean", "std", "min", "50%", "max"]]
summary_df.rename(columns={"50%": "median"}, inplace=True)
print("=================== 📊 AGGREGATED METRICS SUMMARY ===================")
summary_df


=================== 📊 AGGREGATED METRICS SUMMARY ===================


In [7]:
# Display Detailed Evaluation Samples
print("=================== 🔍 DETAILED SAMPLE EVALUATIONS ===================")
for idx, row in df_results.head(5).iterrows():
    print(f"\n--- [Q{row['id']}] {row['question']} ---")
    print(f"🎯 Ground Truth : {row['ground_truth_answer']}")
    print(f"🤖 RAG Answer   : {row['rag_generated_answer']}")
    print(f"📊 Metrics      : SemSim={row['semantic_similarity']:.4f} | TokenF1={row['token_f1']:.4f} | JudgeScore={row['llm_judge_score']}/5 | Latency={row['latency_seconds']}s")
    print(f"💡 Judge Reason : {row['llm_judge_reason']}")
